# Feature Selection — Problem B (`delivery_days`)

How features are chosen:

1. **Candidates come from `src/features.py`.** `FEATURES_LINEAR` is the deduplicated list for Lasso (log1p on skewed columns, fixed bands, rare categories grouped), `FEATURES_TREE` the full list for tree models. This notebook selects within them, so the models in `05` use the same definitions.
2. **Target:** `log(delivery_days / promised_delivery_days)`, which removes the drift in delivery time between train, validation and test.
3. **Linear models:** Lasso stability selection: features selected in at least 80% of the time-ordered folds at the CV-best alpha.
4. **Tree models:** permutation importance of a gradient-boosting model on each time-ordered fold; features that help in at least 80% of folds.
5. **Model choice uses time-ordered CV on the training set** (mean ± std over 5 folds). Validation (July 2018) is shown but not used to choose: its late rate (3.4%) and delivery times are unlike the rest of the data. The test set is not used.
6. Every model is compared with a naive baseline: `promised_delivery_days` × the median train ratio.
7. Validation error is split by whether the order's sellers have delivery history, and the seller-history smoothing strength is checked on CV.

## Load dataset

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CV_FOLDS, RANDOM_STATE, TABLES as TABLES_DIR, set_seed
from src.evaluate import regression_metrics
from src.features import (
    FEATURES_LINEAR, FEATURES_TREE, build_preprocessor, feature_columns,
    load_feature_table, output_sources,
)

set_seed()

In [2]:
# Cleaned regression file + Part A and Part B features (run 02_preprocessing first).
# Seller outcome history uses training orders only (the default history_splits).
df = load_feature_table("regression")

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

print("Dataset shape:", df.shape)

Dataset shape: (96454, 72)


## 1. Candidate features

`FEATURES_LINEAR` and `FEATURES_TREE` already exclude IDs, the raw timestamp, `purchase_month` (`"YYYY-MM"`), `split` and both targets. `FEATURES_LINEAR` also leaves out near-duplicate columns; the reason for each is listed in `src/features.py`.

In [3]:
target = "delivery_days"

linear_cols = feature_columns(FEATURES_LINEAR)
tree_cols = feature_columns(FEATURES_TREE)

assert target not in linear_cols + tree_cols
assert "promised_delivery_days" in linear_cols and "promised_delivery_days" in tree_cols

print("Linear candidates:", len(linear_cols))
for group, columns in FEATURES_LINEAR.items():
    print(f"  {group:12s} {len(columns):2d}  {columns}")
print("\nTree candidates:", len(tree_cols))

Linear candidates: 36
  numeric      18  ['purchase_month_sin', 'purchase_month_cos', 'peak_season_flag', 'platform_load_ratio', 'customer_is_capital', 'remote_flag', 'same_state', 'same_city', 'multi_seller_flag', 'used_voucher', 'payment_installments', 'longest_side_cm', 'geolocation_missing', 'payment_missing', 'seller_is_new', 'seller_past_late_rate', 'seller_past_delivery_days', 'route_past_late_rate']
  skewed       11  ['promised_delivery_days', 'estimated_mean_distance_km', 'freight_value', 'order_value', 'item_count', 'total_weight_g', 'total_volume_cm3', 'customer_zip_density', 'seller_past_orders', 'seller_backlog_7d', 'seller_past_avg_freight']
  categorical   5  ['product_category', 'region_pair', 'customer_state', 'payment_type', 'purchase_weekday']
  banded        3  ['days_to_holiday', 'estimated_mean_distance_km', 'purchase_hour']

Tree candidates: 63


## 2. Split and check redundancy

Rows are sorted by purchase time so that every `TimeSeriesSplit` fold trains on earlier orders and is scored on later ones.

The redundancy check uses Spearman (rank) correlation, which is unaffected by the log1p transform and by skew. No pair of numeric linear candidates may exceed 0.90 on the training set.

In [4]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

X_train = train_df[tree_cols + [c for c in linear_cols if c not in tree_cols]].copy()
y_train = train_df[target].copy()

X_val = val_df[X_train.columns].copy()
y_val = val_df[target].copy()

Train: (83947, 72)
Validation: (6156, 72)
Test: (6351, 72)


In [5]:
numeric_linear = list(dict.fromkeys(FEATURES_LINEAR["numeric"] + FEATURES_LINEAR["skewed"]))

corr_matrix = (
    X_train[numeric_linear]
    .corr(method="spearman")
    .abs()
)

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
).stack().sort_values(ascending=False)

print("Most correlated linear candidate pairs (Spearman, train):")
print(upper.head(8).round(3))

assert (upper <= 0.90).all(), upper[upper > 0.90]

Most correlated linear candidate pairs (Spearman, train):
longest_side_cm             total_volume_cm3              0.811
total_weight_g              total_volume_cm3              0.783
same_state                  estimated_mean_distance_km    0.755
seller_past_orders          seller_backlog_7d             0.749
longest_side_cm             total_weight_g                0.679
promised_delivery_days      estimated_mean_distance_km    0.637
same_state                  promised_delivery_days        0.603
estimated_mean_distance_km  freight_value                 0.579
dtype: float64


## 3. Target and baseline

Delivery time falls sharply over time, so a model fitted on the raw target over-predicts later months.
Olist's own promise (`promised_delivery_days`) falls with it, so the model predicts the delivery time **relative to the promise**:

$$y = \log\left(\frac{\text{delivery\_days}}{\text{promised\_delivery\_days}}\right)$$

Predictions are converted back to days before computing metrics. The log also reduces the influence of the few very long deliveries (up to 209 days).

The baseline every model must beat: `promised_delivery_days` × the median train ratio of delivery days to promised days.

In [6]:
# Delivery time drifts down; the promise drifts with it
display(
    df.groupby("split", sort=False)[[target, "promised_delivery_days"]]
    .mean()
    .round(2)
)


def to_log_ratio(days, promised):
    return np.log(days / promised)


def to_days(log_ratio, promised):
    return np.exp(log_ratio) * promised


y_train_ratio = to_log_ratio(y_train, X_train["promised_delivery_days"])

print("Log-ratio target (train):")
print(y_train_ratio.describe().round(3))

,delivery_days,promised_delivery_days
split,,
train,13.19,25.32
validation,8.96,20.33
test,7.73,15.82


Log-ratio target (train):
count    83947.000
mean        -0.818
std          0.579
min         -3.736
25%         -1.177
50%         -0.822
75%         -0.458
max          2.527
dtype: float64


In [7]:
tscv = TimeSeriesSplit(n_splits=CV_FOLDS)

# Purchase months covered by each evaluation fold
display(pd.DataFrame([
    {
        "fold": k + 1,
        "fit_rows": len(fit_idx),
        "eval_from": train_df["order_purchase_timestamp"].iloc[eval_idx[0]].date(),
        "eval_to": train_df["order_purchase_timestamp"].iloc[eval_idx[-1]].date(),
        "eval_mean_days": y_train.iloc[eval_idx].mean().round(2),
    }
    for k, (fit_idx, eval_idx) in enumerate(tscv.split(X_train))
]).set_index("fold"))


def regression_report(name, y_true, y_pred):
    # Bias > 0: predicts slower than actual
    return {"Model": name, **regression_metrics(y_true, y_pred)}


def cv_fold_mae(model, X):
    """MAE in days of `model` (fitted on the log-ratio target) on each time-ordered fold."""
    scores = []
    for fit_idx, eval_idx in tscv.split(X):
        fold_model = clone(model).fit(X.iloc[fit_idx], y_train_ratio.iloc[fit_idx])
        pred = to_days(
            fold_model.predict(X.iloc[eval_idx]),
            X_train["promised_delivery_days"].iloc[eval_idx],
        )
        scores.append(mean_absolute_error(y_train.iloc[eval_idx], pred))
    return np.array(scores)


class PromiseBaseline:
    """Predicts the median train log-ratio for every order, i.e. promised days x median ratio."""

    def fit(self, X, y):
        self.median_ = np.median(y)
        return self

    def predict(self, X):
        return np.full(len(X), self.median_)

    def get_params(self, deep=True):
        return {}

    def set_params(self, **params):
        return self


baseline = PromiseBaseline().fit(X_train, y_train_ratio)
print("Median ratio delivery/promised:", round(np.exp(baseline.median_), 3))

,fit_rows,eval_from,eval_to,eval_mean_days
fold,,,,
1,13992,2017-06-29,2017-10-11,11.61
2,27983,2017-10-11,2017-12-18,14.58
3,41974,2017-12-18,2018-02-23,15.01
4,55965,2018-02-23,2018-04-25,14.67
5,69956,2018-04-25,2018-06-30,10.46


Median ratio delivery/promised: 0.439


## 4. Lasso (linear candidates)

`build_preprocessor(**FEATURES_LINEAR)`: median imputation and scaling for numeric columns, log1p first for skewed ones, fixed bands for distance, hour and days to holiday, and one-hot encoding with categories seen fewer than 20 times grouped together. `best_alpha` minimises the mean CV error over the time-ordered folds.

In [8]:
lasso_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(**FEATURES_LINEAR)),
    ("lasso", LassoCV(cv=tscv, max_iter=20000)),
])

lasso_pipeline.fit(
    X_train[linear_cols],
    y_train_ratio
)

lasso_cv = lasso_pipeline.named_steps["lasso"]
best_alpha = lasso_cv.alpha_

print("Best alpha:", best_alpha)

Best alpha: 0.0014432903618070113


In [9]:
# Lasso coefficients at best alpha, fitted on the full training set
coef_df = pd.DataFrame({
    "feature": lasso_pipeline.named_steps["preprocessor"].get_feature_names_out(),
    "source": output_sources(lasso_pipeline.named_steps["preprocessor"]),
    "coefficient": lasso_cv.coef_,
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("Coefficients are on the log-ratio scale: +0.1 is about 10% slower than the promise ratio.")
print("Non-zero at best alpha:", (lasso_cv.coef_ != 0).sum(), "of", len(lasso_cv.coef_))
display(coef_df.head(15))

Coefficients are on the log-ratio scale: +0.1 is about 10% slower than the promise ratio.
Non-zero at best alpha: 57 of 169


,feature,source,coefficient,abs_coefficient
18,promised_delivery_days,promised_delivery_days,-0.229175,0.229175
142,customer_state_SP,customer_state,-0.174561,0.174561
19,estimated_mean_distance_km,estimated_mean_distance_km,0.155552,0.155552
127,customer_state_MG,customer_state,-0.132375,0.132375
160,days_to_holiday_4,days_to_holiday,-0.127442,0.127442
16,seller_past_delivery_days,seller_past_delivery_days,0.116584,0.116584
153,purchase_weekday_4,purchase_weekday,0.098313,0.098313
1,purchase_month_cos,purchase_month_cos,0.087562,0.087562
108,region_pair_SE_to_NE,region_pair,0.086944,0.086944
144,payment_type_boleto,payment_type,0.086739,0.086739


### 4.1 Stability selection

Refit Lasso at `best_alpha` on the training window of each time-ordered fold. An input column counts as selected in a fold if any of its encoded columns (one-hot or band) is non-zero. Keep columns selected in at least 80% of folds.

The previous version of this notebook also tried the one-standard-error alpha (largest alpha within one standard error of the best) for selection. It kept only 16 features and lost to the baseline on validation, so selection uses `best_alpha`.

In [10]:
STABILITY_THRESHOLD = 0.8

linear_counts = pd.Series(0, index=linear_cols)

for fit_idx, _ in tscv.split(X_train):
    fold_model = Pipeline([
        ("preprocessor", build_preprocessor(**FEATURES_LINEAR)),
        ("lasso", Lasso(alpha=best_alpha, max_iter=20000)),
    ]).fit(X_train[linear_cols].iloc[fit_idx], y_train_ratio.iloc[fit_idx])

    sources = np.array(output_sources(fold_model.named_steps["preprocessor"]))
    used = set(sources[fold_model.named_steps["lasso"].coef_ != 0])
    linear_counts[list(used)] += 1

linear_freq = (linear_counts / tscv.get_n_splits()).sort_values(ascending=False)
linear_selected = linear_freq[linear_freq >= STABILITY_THRESHOLD].index.tolist()

# Same grouping as FEATURES_LINEAR, so models can call build_preprocessor(**FEATURES_LINEAR_SELECTED)
FEATURES_LINEAR_SELECTED = {
    group: [c for c in columns if c in linear_selected]
    for group, columns in FEATURES_LINEAR.items()
}

print("Selected in every fold:", int((linear_freq == 1).sum()))
print("Selected in some folds:", int(((linear_freq > 0) & (linear_freq < 1)).sum()))
print("Never selected:", int((linear_freq == 0).sum()))
print(f"\nLinear features kept (>= {STABILITY_THRESHOLD:.0%} of folds):", len(linear_selected), "of", len(linear_cols))
display(linear_freq.to_frame("selection_frequency"))

Selected in every fold: 28
Selected in some folds: 8
Never selected: 0

Linear features kept (>= 80% of folds): 33 of 36


,selection_frequency
purchase_month_sin,1.0
seller_past_delivery_days,1.0
days_to_holiday,1.0
purchase_weekday,1.0
payment_type,1.0
customer_state,1.0
region_pair,1.0
product_category,1.0
seller_backlog_7d,1.0
customer_zip_density,1.0


## 5. Gradient boosting (tree candidates)

Lasso coefficients say nothing about which features a tree model can use (non-linear effects, interactions), so tree features are chosen separately.

On each time-ordered fold, fit `HistGradientBoostingRegressor` on the fold's training window, then measure **permutation importance** on its evaluation window: how much the MAE of the log-ratio rises when one column is shuffled. A column is kept if it helps (importance > 0) in at least 80% of folds and its mean importance is positive. Scoring on later orders, rather than on the rows the model was fitted on, penalises columns whose relationship to the target does not hold over time.

`platform_orders_7d` is excluded from `FEATURES_TREE` in `src/features.py`: it grows with the platform, so validation and test values sit beyond the training range.

In [11]:
def tree_model():
    return Pipeline([
        ("preprocessor", build_preprocessor(**FEATURES_TREE, scale=False)),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE,
        )),
    ])


fold_importance = []

for fit_idx, eval_idx in tscv.split(X_train):
    fitted = tree_model().fit(X_train[tree_cols].iloc[fit_idx], y_train_ratio.iloc[fit_idx])
    result = permutation_importance(
        fitted,
        X_train[tree_cols].iloc[eval_idx],
        y_train_ratio.iloc[eval_idx],
        scoring="neg_mean_absolute_error",
        n_repeats=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    fold_importance.append(pd.Series(result.importances_mean, index=tree_cols))

fold_importance = pd.concat(fold_importance, axis=1)
fold_importance.columns = [f"fold_{k + 1}" for k in range(fold_importance.shape[1])]

tree_summary = pd.DataFrame({
    "mean_importance": fold_importance.mean(axis=1),
    "positive_share": (fold_importance > 0).mean(axis=1),
}).sort_values("mean_importance", ascending=False)

tree_selected = tree_summary[
    (tree_summary["positive_share"] >= STABILITY_THRESHOLD)
    & (tree_summary["mean_importance"] > 0)
].index.tolist()

FEATURES_TREE_SELECTED = {
    group: [c for c in columns if c in tree_selected]
    for group, columns in FEATURES_TREE.items()
}

print(f"Tree features kept (helpful in >= {STABILITY_THRESHOLD:.0%} of folds, positive on average):", len(tree_selected), "of", len(tree_cols))
display(tree_summary.round(4))

Tree features kept (helpful in >= 80% of folds, positive on average): 37 of 63


,mean_importance,positive_share
promised_days_per_km,0.0279,1.0
promised_delivery_days,0.0243,1.0
seller_past_delivery_days,0.0107,1.0
customer_zip_lat,0.0067,1.0
customer_state,0.0053,0.8
...,...,...
same_city,-0.0000,0.4
max_item_price,-0.0000,0.8
peak_season_flag,-0.0000,0.2
primary_seller_region,-0.0001,0.6


## 6. Compare models

Chosen by **CV MAE** over the time-ordered folds of the training set (mean ± std, and each fold below). Validation is shown for reference only. All models predict the log-ratio target and are scored in days.

In [12]:
def lasso_on(features):
    return Pipeline([
        ("preprocessor", build_preprocessor(**features)),
        ("model", Lasso(alpha=best_alpha, max_iter=20000)),
    ])


def tree_on(features):
    model = tree_model()
    model.set_params(preprocessor=build_preprocessor(**features, scale=False))
    return model


candidates = {
    "Baseline: promised x ratio": (baseline, linear_cols),
    f"Linear Regression, {len(linear_cols)} linear features": (
        Pipeline([("preprocessor", build_preprocessor(**FEATURES_LINEAR)), ("model", LinearRegression())]),
        linear_cols,
    ),
    f"Lasso, {len(linear_cols)} linear features": (lasso_on(FEATURES_LINEAR), linear_cols),
    f"Lasso, {len(linear_selected)} selected features": (lasso_on(FEATURES_LINEAR_SELECTED), linear_selected),
    f"Gradient boosting, {len(tree_cols)} tree features": (tree_on(FEATURES_TREE), tree_cols),
    f"Gradient boosting, {len(tree_selected)} selected features": (tree_on(FEATURES_TREE_SELECTED), tree_selected),
}

rows, folds, val_predictions = [], {}, {}
for name, (model, columns) in candidates.items():
    fold_mae = cv_fold_mae(model, X_train[columns])
    folds[name] = fold_mae
    fitted = clone(model).fit(X_train[columns], y_train_ratio)
    val_pred = to_days(fitted.predict(X_val[columns]), X_val["promised_delivery_days"])
    val_predictions[name] = val_pred
    val = regression_report(name, y_val, val_pred)
    rows.append({
        "Model": name,
        "CV MAE mean": fold_mae.mean(),
        "CV MAE std": fold_mae.std(),
        "Val MAE": val["MAE"],
        "Val R2": val["R2"],
        "Val bias": val["Bias"],
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("CV MAE mean")
display(comparison.round(3))

fold_table = pd.DataFrame(folds, index=[f"fold_{k + 1}" for k in range(CV_FOLDS)]).T
display(fold_table.loc[comparison.index].round(3))

,CV MAE mean,CV MAE std,Val MAE,Val R2,Val bias
Model,,,,,
"Gradient boosting, 37 selected features",5.079,0.867,3.624,0.255,1.094
"Gradient boosting, 63 tree features",5.121,0.916,3.585,0.262,1.018
"Lasso, 33 selected features",5.153,0.871,3.275,0.295,0.111
"Lasso, 36 linear features",5.186,0.826,3.332,0.288,0.268
"Linear Regression, 36 linear features",5.236,0.752,3.288,0.289,0.112
Baseline: promised x ratio,5.755,0.952,3.580,0.185,-0.026


,fold_1,fold_2,fold_3,fold_4,fold_5
Model,,,,,
"Gradient boosting, 37 selected features",3.759,5.786,5.604,5.910,4.336
"Gradient boosting, 63 tree features",3.791,5.751,5.849,5.971,4.241
"Lasso, 33 selected features",3.987,5.891,5.630,6.039,4.216
"Lasso, 36 linear features",3.987,5.892,5.621,6.018,4.415
"Linear Regression, 36 linear features",4.260,5.902,5.631,5.989,4.396
Baseline: promised x ratio,4.203,6.315,6.136,6.920,5.199


In [13]:
baseline_name = "Baseline: promised x ratio"
wins = (fold_table.drop(index=baseline_name) < fold_table.loc[baseline_name]).sum(axis=1)

print("Folds (of 5) in which each model beats the baseline:")
print(wins.loc[[m for m in comparison.index if m != baseline_name]].to_string())

Folds (of 5) in which each model beats the baseline:
Gradient boosting, 37 selected features    5
Gradient boosting, 63 tree features        5
Lasso, 33 selected features                5
Lasso, 36 linear features                  5
Linear Regression, 36 linear features      4


### 6.1 Known vs new sellers

Seller-history features fall back to the training average when a seller has no delivered training orders. Such orders are 13.4% of test, so the error is shown separately for them. An order counts as **new seller** if at least one of its sellers has no delivered training history (`seller_past_deliveries == 0`).

In [14]:
new_seller = val_df["seller_past_deliveries"].eq(0)
groups = {"known seller": ~new_seller.to_numpy(), "new seller": new_seller.to_numpy()}

print({group: int(mask.sum()) for group, mask in groups.items()}, "validation orders")

seller_breakdown = pd.DataFrame({
    name: {
        f"{group} MAE": mean_absolute_error(y_val[mask], pred[mask])
        for group, mask in groups.items()
    }
    for name, pred in val_predictions.items()
}).T
seller_breakdown["gap"] = seller_breakdown["new seller MAE"] - seller_breakdown["known seller MAE"]
display(seller_breakdown.loc[comparison.index].round(3))

{'known seller': 5739, 'new seller': 417} validation orders


,known seller MAE,new seller MAE,gap
Model,,,
"Gradient boosting, 37 selected features",3.636,3.469,-0.166
"Gradient boosting, 63 tree features",3.593,3.470,-0.123
"Lasso, 33 selected features",3.304,2.878,-0.425
"Lasso, 36 linear features",3.357,2.988,-0.369
"Linear Regression, 36 linear features",3.313,2.943,-0.370
Baseline: promised x ratio,3.611,3.148,-0.463


On validation, orders with a new seller are not predicted worse than orders with known sellers (only 417 orders, and July is unusually fast), so falling back to the training average costs nothing measurable here. Recheck this on test, where 13.4% of orders have a new seller.

### 6.2 Seller-history smoothing strength

Outcome-based seller features are smoothed towards the training average by `SMOOTHING` pseudo-orders (10 in `src/features.py`). The table rebuilds the features with other strengths and scores the selected Lasso on the same time-ordered folds (gradient boosting is left out to keep the run short). `0` means no smoothing.

In [15]:
smoothing_rows = []
for smoothing in [0, 3, 10, 30, 100]:
    frame = (
        load_feature_table("regression", smoothing=smoothing)
        .set_index("order_id")
        .loc[train_df["order_id"]]
    )
    fold_mae = cv_fold_mae(lasso_on(FEATURES_LINEAR_SELECTED), frame[linear_selected])
    smoothing_rows.append({"smoothing": smoothing, "CV MAE mean": fold_mae.mean(), "CV MAE std": fold_mae.std()})

smoothing_table = pd.DataFrame(smoothing_rows).set_index("smoothing")
display(smoothing_table.round(3))

,CV MAE mean,CV MAE std
smoothing,,
0,5.179,0.876
3,5.159,0.869
10,5.153,0.871
30,5.157,0.876
100,5.171,0.884


All strengths score within 0.03 days of each other, far less than the fold-to-fold spread, and 10 is the best of them. `SMOOTHING = 10` is kept.

## 7. Save the selected feature sets

`regression_selected_features.json` holds both sets grouped as `build_preprocessor` arguments:

```python
selected = json.loads((TABLES_DIR / "regression_selected_features.json").read_text())
build_preprocessor(**selected["linear"])
build_preprocessor(**selected["tree"], scale=False)
```

**Final model.** After choosing on CV, refit on train + validation and score once on test. Rebuild the features with `load_feature_table("regression", history_splits=("train", "validation"))` so July's outcomes count as seller history, as they would in production.

In [16]:
selection = pd.DataFrame({
    "linear_selection_frequency": linear_freq,
    "tree_mean_importance": tree_summary["mean_importance"],
    "tree_positive_share": tree_summary["positive_share"],
}).rename_axis("feature")

selection["linear_candidate"] = selection.index.isin(linear_cols)
selection["linear_selected"] = selection.index.isin(linear_selected)
selection["tree_candidate"] = selection.index.isin(tree_cols)
selection["tree_selected"] = selection.index.isin(tree_selected)

selection = selection.sort_values(["linear_selected", "tree_selected"], ascending=False)
display(selection.round(4))

,linear_selection_frequency,tree_mean_importance,tree_positive_share,linear_candidate,linear_selected,tree_candidate,tree_selected
feature,,,,,,,
customer_is_capital,1.0,0.0004,1.0,True,True,True,True
customer_state,1.0,0.0053,0.8,True,True,True,True
customer_zip_density,1.0,0.0005,0.8,True,True,True,True
freight_value,1.0,0.0019,0.8,True,True,True,True
longest_side_cm,1.0,0.0002,0.8,True,True,True,True
...,...,...,...,...,...,...,...
seller_state_count,NaN,0.0000,0.6,False,False,True,False
total_volume_cm3,0.4,0.0000,0.6,True,False,True,False
used_voucher,0.6,0.0000,0.6,True,False,True,False


In [17]:
table_path = TABLES_DIR / "regression_selected_features.csv"
json_path = TABLES_DIR / "regression_selected_features.json"

selection.reset_index().to_csv(table_path, index=False)
json_path.write_text(json.dumps(
    {"linear": FEATURES_LINEAR_SELECTED, "tree": FEATURES_TREE_SELECTED}, indent=2
) + "\n")

print("Saved to:", table_path.relative_to(ROOT))
print("Saved to:", json_path.relative_to(ROOT))

Saved to: report/tables/regression_selected_features.csv
Saved to: report/tables/regression_selected_features.json
